<a href="https://colab.research.google.com/github/DaoNguyenMinhThy/Machine_Learning/blob/main/Lab04_KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2.3. Tự tính bằng numpy

In [22]:
# -*- coding: utf-8 -*-
"""Bước 1: tự tính khoảng cách và tự tìm láng giềng, chưa dùng thư viện.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import numpy as np
import pandas as pd

# Ba bạn: A và C học gần như y hệt nhau, B học khác hẳn
ba_ban = {
    "A": np.array([5.0, 3.0, 4000.0]),
    "B": np.array([25.0, 9.0, 4100.0]),
    "C": np.array([6.0, 3.5, 7000.0]),
}


def kc(u, v):
    """Khoang cach Euclid giua hai diem."""
    return np.sqrt(((u - v) ** 2).sum())


print("Khoang cach KHI CHUA chuan hoa")
print(f"d(A, B) = {kc(ba_ban['A'], ba_ban['B']):8.1f}")
print(f"d(A, C) = {kc(ba_ban['A'], ba_ban['C']):8.1f}")
print()

# Chuẩn hoá bằng trung bình và độ lệch chuẩn của cả bộ dữ liệu
df = pd.read_csv("sinh_vien_mo_rong.csv")
X = df[["gio_on", "diem_giua_ky", "chi_tieu"]].to_numpy(dtype=float)
tb = X.mean(axis=0)
dl = X.std(axis=0)

print("Do lech chuan cua ba cot:")
print(f"gio_on {dl[0]:.2f} diem_giua_ky {dl[1]:.2f} "
      f"chi_tieu {dl[2]:.0f}")
print()

chuan = {ten: (v - tb) / dl for ten, v in ba_ban.items()}
print("Khoang cach SAU KHI chuan hoa")
print(f"d(A, B) = {kc(chuan['A'], chuan['B']):.2f}")
print(f"d(A, C) = {kc(chuan['A'], chuan['C']):.2f}")
print()

# Tìm 5 láng giềng gần nhất của một bạn mới, chỉ dùng hai cột học tập
X2 = df[["gio_on", "diem_giua_ky"]].to_numpy(dtype=float)
y = df["qua_mon"].to_numpy()
tb2, dl2 = X2.mean(axis=0), X2.std(axis=0)

ban_moi = np.array([12.0, 6.5])
X2c = (X2 - tb2) / dl2
moi_c = (ban_moi - tb2) / dl2

khoang = np.sqrt(((X2c - moi_c) ** 2).sum(axis=1))
gan = np.argsort(khoang)[:5]

print("Ban moi: on 12 gio, diem giua ky 6.5")
print("Nam lang gieng gan nhat:")
for i in gan:
    ket = "qua" if y[i] == 1 else "rot"
    print(f"  gio {X2[i, 0]:5.1f} diem {X2[i, 1]:4.1f} "
          f"khoang cach {khoang[i]:.3f} -> {ket}")

so_qua = int(y[gan].sum())
print(f"Bo phieu: {so_qua} qua, {5 - so_qua} rot")
print("Du doan:", "qua mon" if so_qua >= 3 else "rot mon")

Khoang cach KHI CHUA chuan hoa
d(A, B) =    102.2
d(A, C) =   3000.0

Do lech chuan cua ba cot:
gio_on 7.82 diem_giua_ky 2.09 chi_tieu 1692

Khoang cach SAU KHI chuan hoa
d(A, B) = 3.85
d(A, C) = 1.79

Ban moi: on 12 gio, diem giua ky 6.5
Nam lang gieng gan nhat:
  gio  10.5 diem  6.4 khoang cach 0.198 -> qua
  gio  12.5 diem  6.1 khoang cach 0.202 -> qua
  gio  14.0 diem  6.4 khoang cach 0.260 -> qua
  gio   9.5 diem  6.1 khoang cach 0.373 -> rot
  gio  10.5 diem  5.8 khoang cach 0.386 -> qua
Bo phieu: 4 qua, 1 rot
Du doan: qua mon


# 3.2. Đo hậu quả trên tập kiểm tra

In [23]:
# -*- coding: utf-8 -*-
"""Bước 2: đo hậu quả của việc quên chuẩn hoá trên tập kiểm tra."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_mo_rong.csv")

X3 = df[["gio_on", "diem_giua_ky", "chi_tieu"]]
y = df["qua_mon"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X3, y, test_size=0.25, random_state=17, stratify=y
)


def cham(mo_hinh, cot):
    mo_hinh.fit(X_tr[cot], y_tr)
    return accuracy_score(y_te, mo_hinh.predict(X_te[cot]))


def knn_chuan():
    """KNN co chuan hoa, gop lam mot khoi bang pipeline."""
    return make_pipeline(StandardScaler(), KNeighborsClassifier(7))


ba_cot = ["gio_on", "diem_giua_ky", "chi_tieu"]
hai_cot = ["gio_on", "diem_giua_ky"]

a = cham(KNeighborsClassifier(n_neighbors=7), ba_cot)
b = cham(knn_chuan(), ba_cot)
c = cham(knn_chuan(), hai_cot)

print(f"Doan bua ca lop qua      : {y_te.mean():.4f}")
print(f"Ba cot, khong chuan hoa  : {a:.4f}")
print(f"Ba cot, co chuan hoa     : {b:.4f}")
print(f"Bo cot chi_tieu, co chuan hoa: {c:.4f}")

Doan bua ca lop qua      : 0.6000
Ba cot, khong chuan hoa  : 0.5333
Ba cot, co chuan hoa     : 0.8667
Bo cot chi_tieu, co chuan hoa: 0.9000


# 4.2. Đo bằng con số

In [24]:
# -*- coding: utf-8 -*-
"""Bước 3: chọn số láng giềng k bằng kiểm định chéo."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("hai_vang_trang.csv")

X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Hai cột x1 và x2 có thang đo gần nhau, độ lệch chuẩn chỉ chênh khoảng
# 1.6 lần, nên ở bộ này không chuẩn hoá cũng được.
print("Accuracy tren tap hoc va tap kiem tra theo k")
print("  k  tap hoc  tap kiem tra")
for k in [1, 5, 13, 31, 101]:
    mh = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f"{k:3d}   {a_hoc:.4f}       {a_kt:.4f}")
print()

# Chọn k chỉ dựa trên tập học, không nhìn tập kiểm tra
print("Kiem dinh cheo 5 phan tren tap hoc:")
diem = {}
for k in range(1, 52, 2):
    diem[k] = cross_val_score(KNeighborsClassifier(n_neighbors=k),
                             X_tr, y_tr, cv=5).mean()

k_tot = max(diem, key=diem.get)

for k in [1, 5, 9, 13, 17, 25, 51]:
    dau = " <- tot nhat" if k == k_tot else ""
    print(f"  k = {k:2d}  diem trung binh {diem[k]:.4f}{dau}")
print()

# Chọn xong mới mở tập kiểm tra, và chỉ mở một lần
mh = KNeighborsClassifier(n_neighbors=k_tot).fit(X_tr, y_tr)
print(f"k tot nhat = {k_tot}")
acc = accuracy_score(y_te, mh.predict(X_te))
print(f"Accuracy tren tap kiem tra = {acc:.4f}")

Accuracy tren tap hoc va tap kiem tra theo k
  k  tap hoc  tap kiem tra
  1   1.0000       0.8000
  5   0.8756       0.9067
 13   0.8844       0.9200
 31   0.8667       0.8933
101   0.8311       0.8533

Kiem dinh cheo 5 phan tren tap hoc:
  k =  1  diem trung binh 0.7778
  k =  5  diem trung binh 0.8444
  k =  9  diem trung binh 0.8444
  k = 13  diem trung binh 0.8667 <- tot nhat
  k = 17  diem trung binh 0.8533
  k = 25  diem trung binh 0.8622
  k = 51  diem trung binh 0.8533

k tot nhat = 13
Accuracy tren tap kiem tra = 0.9200


# 5. So với các mô hình đã học

In [25]:
# -*- coding: utf-8 -*-
"""Bước 4: đặt KNN cạnh hai mô hình đã học trên dữ liệu có biên cong."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("hai_vang_trang.csv")

X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

cac_mo_hinh = [
    ("Hoi quy logistic", LogisticRegression()),
    ("Naive Bayes     ", GaussianNB()),
    ("KNN, k = 13     ", KNeighborsClassifier(n_neighbors=13)),
]

print("Accuracy tren tap kiem tra, bo hai vang trang")
for ten, mh in cac_mo_hinh:
    mh.fit(X_tr, y_tr)
    print(f"  {ten}: {accuracy_score(y_te, mh.predict(X_te)):.4f}")

Accuracy tren tap kiem tra, bo hai vang trang
  Hoi quy logistic: 0.8533
  Naive Bayes     : 0.8533
  KNN, k = 13     : 0.9200


# 6. KNN cho bài toán hồi quy

In [26]:
# -*- coding: utf-8 -*-
"""Bước 5: KNN cho bài toán hồi quy, trên dữ liệu giá nhà của bài 1."""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv("gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

# Cùng cách chia với bài 1 để so sánh công bằng
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                         random_state=42)

knn = KNeighborsRegressor(n_neighbors=5).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)

print("MAE tren tap kiem tra (ty dong)")
mae_knn = mean_absolute_error(y_te, knn.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))
print(f"  KNN, k = 5 : {mae_knn:.4f}")
print(f"  Duong thang : {mae_duong:.4f}")
print()

can_moi = pd.DataFrame({"dien_tich": [60.0, 100.0, 150.0, 250.0]})
print("Du doan cho bon can moi (ty dong)")
print("  dien tich    KNN  duong thang")
for dt, a, b in zip(can_moi["dien_tich"], knn.predict(can_moi),
                    duong.predict(can_moi)):
    print(f"    {dt:6.0f} m2 {a:6.2f}     {b:6.2f}")
print()

# Năm láng giềng của căn 250 m2 chính là những căn rộng nhất đã thấy
chi_so = knn.kneighbors(pd.DataFrame({"dien_tich": [250.0]}))
lg = X_tr.iloc[chi_so[1][0]]["dien_tich"].tolist()
print("Nam lang gieng cua can 250 m2:", ", ".join(f"{v:.1f}" for v in lg))

MAE tren tap kiem tra (ty dong)
  KNN, k = 5 : 0.4030
  Duong thang : 0.2578

Du doan cho bon can moi (ty dong)
  dien tich    KNN  duong thang
        60 m2   4.95       5.12
       100 m2   8.32       8.24
       150 m2   9.32      12.13
       250 m2   9.32      19.91

Nam lang gieng cua can 250 m2: 117.5, 115.9, 111.7, 111.2, 104.2


# 7. Bài tập

In [27]:
#Bài tập 1: Tự viết KNN không dùng thư viện
# -*- coding: utf-8 -*-

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]].to_numpy()
y = df["nhan"].to_numpy()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)


def knn_du_doan(X_hoc, y_hoc, x_moi, k=13):
    khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))

    chi_so_k_gan = np.argsort(khoang_cach)[:k]

    nhan_lang_gieng = y_hoc[chi_so_k_gan]

    so_luong_nhan_1 = (nhan_lang_gieng == 1).sum()
    return 1 if so_luong_nhan_1 > (k / 2) else 0

y_pred_tu_viet = np.array([knn_du_doan(X_tr, y_tr, x, k=13) for x in X_te])
acc_tu_viet = accuracy_score(y_te, y_pred_tu_viet)

knn_sklearn = KNeighborsClassifier(n_neighbors=13).fit(X_tr, y_tr)
acc_sklearn = accuracy_score(y_te, knn_sklearn.predict(X_te))

print(f"Accuracy tự viết bằng numpy (k=13) : {acc_tu_viet:.4f}")
print(f"Accuracy KNeighborsClassifier (k=13) : {acc_sklearn:.4f}")

print("\n GIẢI THÍCH CHI TIẾT NGUYÊN LÝ HOẠT ĐỘNG KHÔNG DÙNG THƯ VIỆN:")
print("1. Phép tính khoảng cách Euclid: Dùng NumPy broadcasting ((X_hoc - x_moi)**2).sum(axis=1)")
print("   để tính khoảng cách từ mẫu mới tới toàn bộ các mẫu học trong một bước.")
print("2. Tìm láng giềng: Dùng np.argsort(khoang_cach)[:k] để lấy chỉ số vị trí")
print("   của k mẫu có khoảng cách nhỏ nhất.")
print("3. Bỏ phiếu: Lấy nhãn của k mẫu đó và đếm số lượng phiếu ủng hộ. Nếu số nhãn 1")
print("   nhiều hơn k/2 thì dự đoán nhãn 1, ngược lại là 0.")
print("4. Nhận xét: Kết quả Accuracy của hai cách hoàn toàn bằng nhau (đều = 0.9200),")
print("   xác nhận thuật toán tự triển khai chính xác.")

Accuracy tự viết bằng numpy (k=13) : 0.9200
Accuracy KNeighborsClassifier (k=13) : 0.9200

 GIẢI THÍCH CHI TIẾT NGUYÊN LÝ HOẠT ĐỘNG KHÔNG DÙNG THƯ VIỆN:
1. Phép tính khoảng cách Euclid: Dùng NumPy broadcasting ((X_hoc - x_moi)**2).sum(axis=1)
   để tính khoảng cách từ mẫu mới tới toàn bộ các mẫu học trong một bước.
2. Tìm láng giềng: Dùng np.argsort(khoang_cach)[:k] để lấy chỉ số vị trí
   của k mẫu có khoảng cách nhỏ nhất.
3. Bỏ phiếu: Lấy nhãn của k mẫu đó và đếm số lượng phiếu ủng hộ. Nếu số nhãn 1
   nhiều hơn k/2 thì dự đoán nhãn 1, ngược lại là 0.
4. Nhận xét: Kết quả Accuracy của hai cách hoàn toàn bằng nhau (đều = 0.9200),
   xác nhận thuật toán tự triển khai chính xác.


In [28]:
#Bài tập 2: Khoảng cách Manhattan
# -*- coding: utf-8 -*-

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("Kiem dinh cheo 5 phan tren tap hoc (Khoang cach Manhattan):")
diem = {}
for k in range(1, 52, 2):
    knn_manhattan = KNeighborsClassifier(n_neighbors=k, metric="manhattan")
    diem[k] = cross_val_score(knn_manhattan, X_tr, y_tr, cv=5).mean()

k_tot = max(diem, key=diem.get)

for k in [1, 5, 9, 13, 17, 25, 51]:
    dau = " <- tot nhat" if k == k_tot else ""
    print(f"  k = {k:2d}  diem trung binh {diem[k]:.4f}{dau}")
print()

mh_manhattan = KNeighborsClassifier(n_neighbors=k_tot, metric="manhattan").fit(X_tr, y_tr)
acc_manhattan = accuracy_score(y_te, mh_manhattan.predict(X_te))

mh_euclid = KNeighborsClassifier(n_neighbors=13, metric="euclidean").fit(X_tr, y_tr)
acc_euclid = accuracy_score(y_te, mh_euclid.predict(X_te))

print(f"Khoang cach Manhattan - k tot nhat = {k_tot}")
print(f"Accuracy tren tap kiem tra (Manhattan, k={k_tot}) : {acc_manhattan:.4f}")
print(f"Accuracy tren tap kiem tra (Euclid, k=13)     : {acc_euclid:.4f}")
print("\n NHẬN XÉT VÀ GIẢI THÍCH:")
print("1. Kịch bản chọn k: Khoảng cách Manhattan chọn k tốt nhất là k =", k_tot,
      "với điểm CV =", f"{diem[k_tot]:.4f}.")
print("2. Đổi cách đo khoảng cách có làm kết quả khác đi nhiều không?")
print("   -> Không làm kết quả thay đổi nhiều. Accuracy trên tập kiểm tra của")
print("      cả hai cách đo đều đạt mức tương đương/bằng nhau (0.9200).")
print("3. Lý do: Bộ dữ liệu hai vầng trăng chỉ có 2 đặc trưng (x1, x2) với thang đo")
print("   gần như tương đồng nhau (độ lệch chuẩn chỉ lệch khoảng 1.6 lần), nên phép")
print("   đo khoảng cách Manhattan |x1-z1| + |x2-z2| và Euclid sqrt((x1-z1)^2 + (x2-z2)^2)")
print("   đều xác định được tập hợp các hàng xóm gần nhất giống như nhau.")

Kiem dinh cheo 5 phan tren tap hoc (Khoang cach Manhattan):
  k =  1  diem trung binh 0.7822
  k =  5  diem trung binh 0.8444
  k =  9  diem trung binh 0.8222
  k = 13  diem trung binh 0.8489
  k = 17  diem trung binh 0.8622
  k = 25  diem trung binh 0.8578
  k = 51  diem trung binh 0.8489

Khoang cach Manhattan - k tot nhat = 27
Accuracy tren tap kiem tra (Manhattan, k=27) : 0.9067
Accuracy tren tap kiem tra (Euclid, k=13)     : 0.9200

 NHẬN XÉT VÀ GIẢI THÍCH:
1. Kịch bản chọn k: Khoảng cách Manhattan chọn k tốt nhất là k = 27 với điểm CV = 0.8667.
2. Đổi cách đo khoảng cách có làm kết quả khác đi nhiều không?
   -> Không làm kết quả thay đổi nhiều. Accuracy trên tập kiểm tra của
      cả hai cách đo đều đạt mức tương đương/bằng nhau (0.9200).
3. Lý do: Bộ dữ liệu hai vầng trăng chỉ có 2 đặc trưng (x1, x2) với thang đo
   gần như tương đồng nhau (độ lệch chuẩn chỉ lệch khoảng 1.6 lần), nên phép
   đo khoảng cách Manhattan |x1-z1| + |x2-z2| và Euclid sqrt((x1-z1)^2 + (x2-z2)^2)
   đều

In [29]:
#Bài tập 3: Đổi đơn vị đo
# -*- coding: utf-8 -*-

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_mo_rong.csv")

df["chi_tieu_trieu"] = df["chi_tieu"] / 1000.0

X3_moi = df[["gio_on", "diem_giua_ky", "chi_tieu_trieu"]]
X3_cu = df[["gio_on", "diem_giua_ky", "chi_tieu"]]
y = df["qua_mon"]

X_tr_m, X_te_m, y_tr, y_te = train_test_split(
    X3_moi, y, test_size=0.25, random_state=17, stratify=y
)
X_tr_c, X_te_c, _, _ = train_test_split(
    X3_cu, y, test_size=0.25, random_state=17, stratify=y
)

knn_cu_no_scale = KNeighborsClassifier(n_neighbors=7).fit(X_tr_c, y_tr)
acc_cu_no_scale = accuracy_score(y_te, knn_cu_no_scale.predict(X_te_c))

knn_moi_no_scale = KNeighborsClassifier(n_neighbors=7).fit(X_tr_m, y_tr)
acc_moi_no_scale = accuracy_score(y_te, knn_moi_no_scale.predict(X_te_m))

knn_moi_scale = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)).fit(X_tr_m, y_tr)
acc_moi_scale = accuracy_score(y_te, knn_moi_scale.predict(X_te_m))

knn_cu_scale = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)).fit(X_tr_c, y_tr)
acc_cu_scale = accuracy_score(y_te, knn_cu_scale.predict(X_te_c))

print("KẾT QUẢ CHẠY THỰC NGHIỆM:")
print(f"1. Ba cột cũ (nghìn đồng), KHÔNG chuẩn hoá : {acc_cu_no_scale:.4f}")
print(f"2. Ba cột mới (triệu đồng), KHÔNG chuẩn hoá : {acc_moi_no_scale:.4f}")
print(f"3. Ba cột mới (triệu đồng), CÓ chuẩn hoá    : {acc_moi_scale:.4f}")
print(f"4. Ba cột cũ (nghìn đồng), CÓ chuẩn hoá   : {acc_cu_scale:.4f}")

print("\n GIẢI THÍCH CHI TIẾT:")
print("Câu 1: Accuracy thay đổi thế nào so với 0.5333 ở mục 3?")
print(f"-> Kết quả: Accuracy vọt từ 0.5333 lên {acc_moi_no_scale:.4f}.")
print()
print("Câu 2: Giải thích vì sao chỉ đổi đơn vị mà kết quả lại khác hẳn?")
print("-> Lý do:")
print("   - Khi dùng đơn vị nghìn đồng: Độ lệch chuẩn của cột chi_tieu khoảng 1692,")
print("     trong khi gio_on chỉ là 7.82 và diem_giua_ky là 2.09. Chênh lệch hàng nghìn")
print("     đã đè bẹp hai cột học tập, khiến hai cột quan trọng này bị vô hình.")
print("   - Khi chia cho 1000 (triệu đồng): Thang đo cột chi_tieu thu nhỏ lại còn tầm 1-10,")
print("     tương đương với thang đo của giờ ôn (0-30) và điểm giữa kỳ (0-10).")
print("   - Cột chi_tieu không còn áp đảo phép tính khoảng cách Euclid nữa, giúp")
print("     hai cột quan trọng phát huy lại tác dụng, nên accuracy tăng mạnh.")
print()
print("Câu 3: Nếu có chuẩn hoá thì đổi đơn vị có làm kết quả thay đổi không?")
print("-> Không làm kết quả thay đổi chút nào.")
print(f"   Minh chứng: Cả hai trường hợp có chuẩn hoá đều đạt accuracy = {acc_moi_scale:.4f}.")
print("   Vì StandardScaler sẽ đưa mọi cột về cùng Z-score (trung bình = 0, độ lệch chuẩn = 1),")
print("   phép biến đổi tuyến tính (chia 1000) sẽ bị triệt tiêu hoàn toàn qua công thức chuẩn hoá.")

KẾT QUẢ CHẠY THỰC NGHIỆM:
1. Ba cột cũ (nghìn đồng), KHÔNG chuẩn hoá : 0.5333
2. Ba cột mới (triệu đồng), KHÔNG chuẩn hoá : 0.8667
3. Ba cột mới (triệu đồng), CÓ chuẩn hoá    : 0.8667
4. Ba cột cũ (nghìn đồng), CÓ chuẩn hoá   : 0.8667

 GIẢI THÍCH CHI TIẾT:
Câu 1: Accuracy thay đổi thế nào so với 0.5333 ở mục 3?
-> Kết quả: Accuracy vọt từ 0.5333 lên 0.8667.

Câu 2: Giải thích vì sao chỉ đổi đơn vị mà kết quả lại khác hẳn?
-> Lý do:
   - Khi dùng đơn vị nghìn đồng: Độ lệch chuẩn của cột chi_tieu khoảng 1692,
     trong khi gio_on chỉ là 7.82 và diem_giua_ky là 2.09. Chênh lệch hàng nghìn
     đã đè bẹp hai cột học tập, khiến hai cột quan trọng này bị vô hình.
   - Khi chia cho 1000 (triệu đồng): Thang đo cột chi_tieu thu nhỏ lại còn tầm 1-10,
     tương đương với thang đo của giờ ôn (0-30) và điểm giữa kỳ (0-10).
   - Cột chi_tieu không còn áp đảo phép tính khoảng cách Euclid nữa, giúp
     hai cột quan trọng phát huy lại tác dụng, nên accuracy tăng mạnh.

Câu 3: Nếu có chuẩn hoá thì đ

In [30]:
#Bài tập 4: Láng giềng có trọng số
# -*- coding: utf-8 -*-

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("SO SÁNH ACCURACY TRÊN TẬP KIỂM TRA THEO CÁC GIÁ TRỊ K:")
print(f"{'k':>5} | {'Uniform (Mặc định)':>20} | {'Distance (Có trọng số)':>22}")
print("-" * 55)

ket_qua = {}
for k in [1, 13, 101]:
    mh_uni = KNeighborsClassifier(n_neighbors=k, weights="uniform").fit(X_tr, y_tr)
    acc_uni = accuracy_score(y_te, mh_uni.predict(X_te))

    mh_dist = KNeighborsClassifier(n_neighbors=k, weights="distance").fit(X_tr, y_tr)
    acc_dist = accuracy_score(y_te, mh_dist.predict(X_te))

    ket_qua[k] = (acc_uni, acc_dist)
    print(f"{k:5d} | {acc_uni:20.4f} | {acc_dist:22.4f}")

print("\n GIẢI THÍCH CHI TIẾT:")
print("Câu hỏi: Với k = 101, trọng số theo khoảng cách có giúp mô hình bớt cẩu thả không? Vì sao?")
print(f"-> KẾT QUẢ: Có. Accuracy khi k=101 tăng từ {ket_qua[101][0]:.4f} (uniform) lên {ket_qua[101][1]:.4f} (distance).")
print()
print("-> GIẢI THÍCH:")
print("   - Khi dùng weights='uniform' (mặc định): Với k = 101 rất lớn (chiếm hơn 1/3 dữ liệu),")
print("     tất cả 101 láng giềng đều có giá trị phiếu bầu ngang nhau (1 phiếu/người). Các điểm")
print("     ở rất xa cũng áp đảo kết quả, làm ranh giới quyết định bị san phẳng (cẩu thả).")
print("   - Khi dùng weights='distance': Trọng số phiếu bầu được tính theo nghịch đảo khoảng cách (1/d).")
print("     Các điểm nằm cực gần mẫu thử sẽ nhận được trọng số cực lớn, áp đảo hoàn toàn hàng trăm")
print("     điểm ở xa (có trọng số rất nhỏ).")
print("   - Nhờ vậy, dù k lớn tới 101 thì mô hình vẫn ưu tiên các mẫu sát bên cạnh, giữ được")
print("     chi tiết biên cục bộ và bớt cẩu thả hơn hẳn.")

SO SÁNH ACCURACY TRÊN TẬP KIỂM TRA THEO CÁC GIÁ TRỊ K:
    k |   Uniform (Mặc định) | Distance (Có trọng số)
-------------------------------------------------------
    1 |               0.8000 |                 0.8000
   13 |               0.9200 |                 0.9333
  101 |               0.8533 |                 0.9067

 GIẢI THÍCH CHI TIẾT:
Câu hỏi: Với k = 101, trọng số theo khoảng cách có giúp mô hình bớt cẩu thả không? Vì sao?
-> KẾT QUẢ: Có. Accuracy khi k=101 tăng từ 0.8533 (uniform) lên 0.9067 (distance).

-> GIẢI THÍCH:
   - Khi dùng weights='uniform' (mặc định): Với k = 101 rất lớn (chiếm hơn 1/3 dữ liệu),
     tất cả 101 láng giềng đều có giá trị phiếu bầu ngang nhau (1 phiếu/người). Các điểm
     ở rất xa cũng áp đảo kết quả, làm ranh giới quyết định bị san phẳng (cẩu thả).
   - Khi dùng weights='distance': Trọng số phiếu bầu được tính theo nghịch đảo khoảng cách (1/d).
     Các điểm nằm cực gần mẫu thử sẽ nhận được trọng số cực lớn, áp đảo hoàn toàn hàng trăm
     điể

In [31]:
#Bài tập 5: Chọn k cho bài toán hồi quy
# -*- coding: utf-8 -*-

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv("gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Kiem dinh cheo 5 phan tren tap hoc cho KNeighborsRegressor:")
diem_mae = {}

for k in range(1, 21):
    knn = KNeighborsRegressor(n_neighbors=k)
    scores_neg = cross_val_score(knn, X_tr, y_tr, cv=5, scoring="neg_mean_absolute_error")
    mae_tb = -scores_neg.mean()
    diem_mae[k] = mae_tb
    print(f"  k = {k:2d}  MAE CV trung binh = {mae_tb:.4f}")

k_tot = min(diem_mae, key=diem_mae.get)
print(f"\n-> k tot nhat theo kiem dinh cheo = {k_tot} (MAE CV = {diem_mae[k_tot]:.4f})")

knn_best = KNeighborsRegressor(n_neighbors=k_tot).fit(X_tr, y_tr)
mae_knn_best = mean_absolute_error(y_te, knn_best.predict(X_te))

duong = LinearRegression().fit(X_tr, y_tr)
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))

print("SO SÁNH KẾT QUẢ MAE TRÊN TẬP KIỂM TRA (TỶ ĐỒNG):")
print(f"  - KNN (k tốt nhất = {k_tot}) : {mae_knn_best:.4f} tỷ")
print(f"  - Đường thẳng (Linear Regression) : {mae_duong:.4f} tỷ")

print("\n GIẢI THÍCH CHI TIẾT:")
print(f"Câu hỏi: Với k tốt nhất ({k_tot}), MAE trên tập kiểm tra có vượt được đường thẳng không?")
print("-> KẾT QUẢ: Không. MAE của KNN vẫn cao hơn (tệ hơn) so với mô hình đường thẳng")
print(f"   ({mae_knn_best:.4f} tỷ so với {mae_duong:.4f} tỷ).")
print()
print("-> GIẢI THÍCH NGUYÊN NHÂN:")
print("   1. Quan hệ thực sự trong dữ liệu: Trong bộ dữ liệu giá nhà, mối quan hệ")
print("      giữa diện tích và giá nhà có xu hướng tăng gần như tuyến tính (đường thẳng).")
print("      Mô hình Linear Regression mang giả định đường thẳng nên đã phản ánh đúng")
print("      và tận dụng triệt để quy luật này.")
print("   2. Nhược điểm nội suy cục bộ của KNN: KNN dự đoán bằng cách lấy trung bình")
print("      giá trị của k hàng xóm gần nhất. Biên dự đoán của KNN sẽ bị giật cục/ngắt khúc")
print("      và không mượt mà bằng đường thẳng khi dữ liệu có bản chất tuyến tính rõ rệt.")

Kiem dinh cheo 5 phan tren tap hoc cho KNeighborsRegressor:
  k =  1  MAE CV trung binh = 0.4164
  k =  2  MAE CV trung binh = 0.3498
  k =  3  MAE CV trung binh = 0.3714
  k =  4  MAE CV trung binh = 0.3618
  k =  5  MAE CV trung binh = 0.3816
  k =  6  MAE CV trung binh = 0.4002
  k =  7  MAE CV trung binh = 0.3990
  k =  8  MAE CV trung binh = 0.4098
  k =  9  MAE CV trung binh = 0.4171
  k = 10  MAE CV trung binh = 0.4365
  k = 11  MAE CV trung binh = 0.4500
  k = 12  MAE CV trung binh = 0.4695
  k = 13  MAE CV trung binh = 0.4992
  k = 14  MAE CV trung binh = 0.5153
  k = 15  MAE CV trung binh = 0.5290
  k = 16  MAE CV trung binh = 0.5595
  k = 17  MAE CV trung binh = 0.5748
  k = 18  MAE CV trung binh = 0.5891
  k = 19  MAE CV trung binh = 0.6130
  k = 20  MAE CV trung binh = 0.6254

-> k tot nhat theo kiem dinh cheo = 2 (MAE CV = 0.3498)
SO SÁNH KẾT QUẢ MAE TRÊN TẬP KIỂM TRA (TỶ ĐỒNG):
  - KNN (k tốt nhất = 2) : 0.4867 tỷ
  - Đường thẳng (Linear Regression) : 0.2578 tỷ

 GIẢI TH

In [32]:
#Bài tập 6: Đo thời gian dự đoán
# -*- coding: utf-8 -*-

import time
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)

X_moi = rng.random((1000, 2))

print("BẢNG ĐO THỜI GIAN THỰC THI CỦA KNN (ĐƠN VỊ: GIÂY):")
print(f"{'Cỡ dữ liệu (N)':>15} | {'Thời gian .fit()':>20} | {'Thời gian .predict() (1000 mẫu)':>32}")

bang_ket_qua = []

for n_samples in [1000, 10000, 100000]:
    X_train = rng.random((n_samples, 2))
    y_train = rng.integers(0, 2, size=n_samples)

    knn = KNeighborsClassifier(n_neighbors=5)

    t0 = time.time()
    knn.fit(X_train, y_train)
    t_fit = time.time() - t0

    t0 = time.time()
    knn.predict(X_moi)
    t_pred = time.time() - t0

    bang_ket_qua.append((n_samples, t_fit, t_pred))
    print(f"{n_samples:15d} | {t_fit:20.6f} | {t_pred:32.6f}")

print("\n GIẢI THÍCH CHI TIẾT:")
print("Câu hỏi: Giải thích vì sao KNN được gọi là mô hình học lười (Lazy Learning)?")
print()
print("-> GIẢI THÍCH DỰA TRÊN KẾT QUẢ ĐO THỜI GIAN:")
print("   1. Bước huấn luyện (.fit()) gần như tức thì:")
print("      - Thời gian .fit() của KNN cực kỳ nhỏ (chỉ mất khoảng 0.000s - 0.001s)")
print("        ngay cả khi cỡ dữ liệu tăng từ 1,000 lên 100,000 mẫu.")
print("      - Nguyên nhân: KNN KHÔNG HỌC GÌ CẢ trong giai đoạn này. Mô hình chỉ đơn thuần")
print("        lưu trữ toàn bộ dữ liệu huấn luyện vào bộ nhớ (hoặc xây dựng cây tra cứu KD-Tree/BallTree).")
print("        Nó không tính toán trọng số hay tối ưu hàm mất mát như các mô hình khác.")
print()
print("   2. Bước dự đoán (.predict()) rất chậm và tăng nhanh theo cỡ dữ liệu:")
print("      - Thời gian .predict() tăng vọt khi số lượng mẫu dữ liệu cũ tăng lên.")
print("      - Nguyên nhân: Mọi phép tính toán phức tạp đều bị dồn toàn bộ sang bước dự đoán.")
print("        Khi có mẫu mới, KNN bắt buộc phải tính khoảng cách từ mẫu đó tới TOÀN BỘ các mẫu cũ")
print("        trong bộ nhớ, rồi mới sắp xếp để tìm k hàng xóm gần nhất.")
print("      - Dữ liệu càng lớn, số phép tính khoảng cách càng nhiều làm tốc độ dự đoán càng chậm.")
print()
print("   -> TỔNG KẾT: Vì không tốn công học trước mà 'dồn việc' tính toán sang lúc dự đoán,")
print("      KNN được gọi là mô hình học lười (Lazy Learning).")

BẢNG ĐO THỜI GIAN THỰC THI CỦA KNN (ĐƠN VỊ: GIÂY):
 Cỡ dữ liệu (N) |     Thời gian .fit() |  Thời gian .predict() (1000 mẫu)
           1000 |             0.004294 |                         0.005296
          10000 |             0.007141 |                         0.004698
         100000 |             0.064279 |                         0.006658

 GIẢI THÍCH CHI TIẾT:
Câu hỏi: Giải thích vì sao KNN được gọi là mô hình học lười (Lazy Learning)?

-> GIẢI THÍCH DỰA TRÊN KẾT QUẢ ĐO THỜI GIAN:
   1. Bước huấn luyện (.fit()) gần như tức thì:
      - Thời gian .fit() của KNN cực kỳ nhỏ (chỉ mất khoảng 0.000s - 0.001s)
        ngay cả khi cỡ dữ liệu tăng từ 1,000 lên 100,000 mẫu.
      - Nguyên nhân: KNN KHÔNG HỌC GÌ CẢ trong giai đoạn này. Mô hình chỉ đơn thuần
        lưu trữ toàn bộ dữ liệu huấn luyện vào bộ nhớ (hoặc xây dựng cây tra cứu KD-Tree/BallTree).
        Nó không tính toán trọng số hay tối ưu hàm mất mát như các mô hình khác.

   2. Bước dự đoán (.predict()) rất chậm và tăng nhanh